In [0]:
%pip install shap

In [0]:
dbutils.library.restartPython()

In [0]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import functions as F

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score, average_precision_score

In [0]:
source_c = spark.table("default.groww_model_c_split")

display(
    source_c.groupBy("dataset_split")
    .agg(
        F.count("*").alias("users"),
        F.sum("target").alias("high_value_users")
    )
)

In [0]:
excluded_cols = [
    "third_party_id",
    "signup_ts",
    "target",
    "dataset_split"
]

feature_cols = [
    c for c in source_c.columns
    if c not in excluded_cols
]

numeric_types = [
    "int", "bigint", "double", "float", "long",
    "decimal", "smallint"
]

numeric_cols = [
    field.name
    for field in source_c.schema.fields
    if any(t in field.dataType.simpleString() for t in numeric_types)
    and field.name in feature_cols
]

categorical_cols = [
    c for c in feature_cols
    if c not in numeric_cols
]

pdf = source_c.select(
    "third_party_id",
    "signup_ts",
    "target",
    "dataset_split",
    *feature_cols
).toPandas()

for col in numeric_cols:
    pdf[col] = pd.to_numeric(pdf[col], errors="coerce")

train_c = pdf[pdf["dataset_split"] == "TRAIN"].copy()
validation_c = pdf[pdf["dataset_split"] == "VALIDATION"].copy()
test_c = pdf[pdf["dataset_split"] == "TEST"].copy()

print("Train:", train_c.shape)
print("Validation:", validation_c.shape)
print("Test:", test_c.shape)

In [0]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

weekly_lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

X_train = train_c[feature_cols]
y_train = train_c["target"].astype(int)

weekly_lr_model.fit(X_train, y_train)

print("Model C trained successfully.")

In [0]:
X_test = test_c[feature_cols]

X_test_transformed = (
    weekly_lr_model
    .named_steps["preprocessor"]
    .transform(X_test)
)

feature_names = (
    weekly_lr_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

classifier = weekly_lr_model.named_steps["classifier"]

# Use a sample of training data as the SHAP background
X_background = (
    weekly_lr_model
    .named_steps["preprocessor"]
    .transform(X_train.sample(
        min(100, len(X_train)),
        random_state=42
    ))
)

explainer = shap.LinearExplainer(
    classifier,
    X_background
)

shap_values = explainer(X_test_transformed)

print("SHAP analysis completed.")

In [0]:
shap.summary_plot(
    shap_values.values,
    X_test_transformed.toarray()
        if hasattr(X_test_transformed, "toarray")
        else X_test_transformed,
    feature_names=feature_names,
    max_display=20
)

In [0]:
X_test = test_c[feature_cols]

X_test_transformed = (
    weekly_lr_model
    .named_steps["preprocessor"]
    .transform(X_test)
)

feature_names = (
    weekly_lr_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

classifier = weekly_lr_model.named_steps["classifier"]

X_background = (
    weekly_lr_model
    .named_steps["preprocessor"]
    .transform(
        X_train.sample(min(100, len(X_train)), random_state=42)
    )
)

explainer = shap.LinearExplainer(
    classifier,
    X_background
)

shap_values = explainer(X_test_transformed)

print("SHAP analysis completed.")
print("Transformed test shape:", X_test_transformed.shape)
print("Number of SHAP features:", len(feature_names))

In [0]:
X_plot = (
    X_test_transformed.toarray()
    if hasattr(X_test_transformed, "toarray")
    else X_test_transformed
)

shap.summary_plot(
    shap_values.values,
    X_plot,
    feature_names=feature_names,
    max_display=20
)

In [0]:
import pandas as pd
import numpy as np

# Convert SHAP values into a DataFrame
shap_df = pd.DataFrame(
    shap_values.values,
    columns=feature_names
)

# Map encoded columns back to their original feature
feature_groups = {}

for col in feature_names:
    if col.startswith("numeric__"):
        original = col.replace("numeric__", "", 1)

    elif col.startswith("categorical__"):
        encoded_name = col.replace("categorical__", "", 1)

        matches = [
            c for c in categorical_cols
            if encoded_name.startswith(c + "_") or encoded_name == c
        ]

        original = max(matches, key=len) if matches else encoded_name

    else:
        original = col

    feature_groups.setdefault(original, []).append(col)

# Aggregate each original feature's SHAP contributions
grouped_importance = []

for original, encoded_cols in feature_groups.items():
    combined_contribution = shap_df[encoded_cols].sum(axis=1)

    grouped_importance.append({
        "feature": original,
        "mean_abs_shap": combined_contribution.abs().mean(),
        "mean_shap": combined_contribution.mean()
    })

shap_importance = (
    pd.DataFrame(grouped_importance)
    .sort_values("mean_abs_shap", ascending=False)
)

display(shap_importance.head(20))

In [0]:
top_features = shap_importance.head(15).sort_values(
    "mean_abs_shap"
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["feature"],
    top_features["mean_abs_shap"]
)

plt.xlabel("Mean Absolute SHAP Value")
plt.title("Top 15 Business Features Driving High-Value Predictions")
plt.tight_layout()
plt.show()

In [0]:
direction_table = (
    shap_importance
    .sort_values("mean_abs_shap", ascending=False)
    [["feature", "mean_abs_shap", "mean_shap"]]
    .head(20)
)

display(direction_table)

In [0]:
# Build a row-level SHAP contribution table
row_shap = pd.DataFrame(
    shap_values.values,
    columns=feature_names,
    index=test_c.index
)

# Reconstruct original-feature SHAP contributions
original_shap = pd.DataFrame(index=test_c.index)

for original, encoded_cols in feature_groups.items():
    original_shap[original] = row_shap[encoded_cols].sum(axis=1)

# Examine categorical and binary features
features_to_check = [
    "income",
    "kyc_at_signup",
    "kyc_tier",
    "aof_within_7d",
    "esign_within_7d",
    "stock_onboarding_within_7d",
    "kra_flag",
    "acquisition_channel",
    "stocks_revenue_7d"
]

for feature in features_to_check:
    if feature not in test_c.columns:
        continue

    analysis_df = pd.DataFrame({
        "value": test_c[feature],
        "shap": original_shap[feature]
    })

    if analysis_df["value"].nunique() > 15:
        analysis_df["value"] = pd.qcut(
            analysis_df["value"],
            q=5,
            duplicates="drop"
        )

    result = (
        analysis_df.groupby("value", observed=True)
        .agg(
            users=("shap", "size"),
            mean_shap=("shap", "mean"),
            mean_abs_shap=("shap", lambda x: x.abs().mean())
        )
        .reset_index()
    )

    print(f"\nFeature: {feature}")
    display(result)